# Creating query for gold customer dimension table

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY c.customer_id) AS customer_key,
    c.customer_id,
    c.customer_number,
    c.first_name,
    c.last_name,
    c.marital_status,
    cl.country,
    CASE
        WHEN c.gender <> 'N/A' THEN c.gender
        ELSE COALESCE(e.gender, 'N/A')
    END AS gender,
    e.birth_date,
    c.created_date
FROM workspace.silver.crm_customers c
LEFT JOIN workspace.silver.erp_customers e
    ON c.customer_number = e.customer_number
LEFT JOIN workspace.silver.erp_customer_location cl
    ON c.customer_number = cl.customer_number
"""

df = spark.sql(query)

In [0]:
df.limit(10).display()

# Writing to Gold table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")

### Sanity check gold table

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customers LIMIT 10